In [0]:
%pip install databricks-feature-engineering


In [0]:
dbutils.library.restartPython()

In [0]:
from databricks.feature_engineering import FeatureEngineeringClient, FeatureLookup

def import_query(path):
    with open(path) as open_file:
        return open_file.read()

# Só é possível porque criamos as feature stores com a biblioteca específica do databricks
# Estamos fazendo um left join das tabelas de uma forma mais rastreável

# Na criação das tabelas eu registrei as chaves primárias nessa ordem, aí para fazer o lookup
# tem que seguir a mesma ordem de chaves (Primary Key 1 and Primary Key 2)
lookups = [
    FeatureLookup(table_name="feature_store.upsell.fs_geral", lookup_key=['IdCliente', 'dtRef']),
    FeatureLookup(table_name="feature_store.upsell.fs_pontos", lookup_key=['IdCliente', 'dtRef']),
    FeatureLookup(table_name="feature_store.upsell.fs_transacoes", lookup_key=['IdCliente', 'dtRef']),
    FeatureLookup(table_name="feature_store.upsell.fs_dia_horario", lookup_key=['IdCliente', 'dtRef']),
]
query = import_query("fl_churn.sql")
df = spark.sql(query)

fe = FeatureEngineeringClient()
training_set = fe.create_training_set(df=df, feature_lookups=lookups, label='flChurn')
df_train = training_set.load_df()

In [0]:
df_train_pandas = df_train.toPandas()
df_train_pandas.head()